# Spatial Topic Comparison

This notebook joins point-capable service-request records to real community-district boundaries and compares complaint mix by joined district.

The goal is to show the reusable `nyc311.spatial` helpers directly on a small offline dataset, then place the joined points on top of a real basemap.

In [ ]:
from pathlib import Path
import sys

import nyc311
from IPython.display import display

repo_root = Path(nyc311.__file__).resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples.utils import data_path, save_boundary_preview

records = nyc311.load_service_requests(data_path("service_requests_fixture.csv"))
records_gdf = nyc311.records_to_geodataframe(records)
boundaries_gdf = nyc311.load_boundaries_geodataframe(
    data_path("community_district_boundaries.geojson")
)
joined = nyc311.spatial_join_records_to_boundaries(records_gdf, boundaries_gdf)
joined.head()

In [ ]:
comparison = (
    joined.dropna(subset=["boundary_geography_value"])
    .groupby(["boundary_geography_value", "complaint_type"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["boundary_geography_value", "count"], ascending=[True, False])
)

display(comparison)

In [ ]:
top_by_district = comparison.groupby("boundary_geography_value").head(3)
display(top_by_district)

preview_path = save_boundary_preview(
    boundaries_gdf,
    points_gdf=records_gdf,
    filename="spatial-topic-comparison-preview.png",
    title="Joined service-request points by community district",
)
preview_path